In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd

from src.scryfall_http_service import ScryfallHttpService
from src.scryfall_data_service import ScryfallDataService
from src.cache_service import CacheService, CacheType

pd.set_option('display.max_columns', None)

os.getcwd()

'c:\\Users\\tallo\\Documents\\programming_projects\\mtg-coding'

In [ ]:
def get_commanders_from_txt_file(fpath: str) -> list[str]:
    with open(fpath, 'r') as f:
        return [x.strip() for x in f.read().split("\n")]


In [3]:
sfds = ScryfallDataService(
    base_url="https://api.scryfall.com/", 
    json_cache_service=CacheService("./cache/json/", cache_type=CacheType.JSON), 
    png_cache_service=CacheService("./cache/png/", cache_type=CacheType.PNG), 
    http_client=ScryfallHttpService(),
)

# sfds.clear_caches()

In [19]:
cards_df = sfds.get_cards_from_bulk_data('gameplay_columns')
# commander_names = cards_df.sample(n=20).name.tolist()
# commander_names = get_commanders_from_txt_file("./data/lists/phils_commanders.txt")
# commander_dfs = cards_df[cards_df.name.isin(commander_names)]
commander_dfs = cards_df.copy()
# commander_dfs.head()

In [ ]:
import re
def count_bools(*arr: list[bool]) -> int:
    cnt = 0
    for x in arr:
        cnt = cnt + (1 if x else 0)
    return cnt

def transform_type_line(df: pd.DataFrame) -> pd.DataFrame:
    df['is_legendary'] = df['type_line'].apply(lambda x: True if len(re.findall("^Legendary", x)) else False)
    df['is_creature'] = df['type_line'].apply(lambda x: True if len(re.findall("Creature", x)) else False)
    df['is_instant'] = df['type_line'].apply(lambda x: True if len(re.findall("Instant", x)) else False)
    df['is_sorcery'] = df['type_line'].apply(lambda x: True if len(re.findall("Sorcery", x)) else False)
    df['is_land'] = df['type_line'].apply(lambda x: True if len(re.findall("Land", x)) else False)
    df['is_enchantment'] = df['type_line'].apply(lambda x: True if len(re.findall("Enchantment", x)) else False)
    df['is_planeswalker'] = df['type_line'].apply(lambda x: True if len(re.findall("Planeswalker", x)) else False)
    df['num_types'] = df.apply(lambda r: count_bools(r['is_creature'], r['is_instant'], r['is_sorcery'], r['is_land'], r['is_enchantment'], r['is_planeswalker']), axis=1)
    return df
# commander_dfs.type_line.apply(lambda x: [i.strip() for i in x.split("—")])

commander_dfs = commander_dfs \
    .pipe(transform_type_line)

# commander_dfs[commander_dfs.is_land & commander_dfs.is_creature].head()
commander_dfs[commander_dfs.num_types.ge(2)].head()

,id,oracle_id,name,cmc,mana_cost,color_identity,colors,type_line,oracle_text,power,toughness,keywords,game_changer,defense,hand_modifier,life_modifier,loyalty,produced_mana,reserved,commander_legality,price_usd,penny_rank,edhrec_rank,is_legendary,is_creature,is_instant,is_sorcery,is_land,is_enchantment,is_planeswalker,num_types
22134,0f503360-216a-4629-89b2-d32072850aef,947b40a2-e272-4ef6-85fb-3d767a21bc7a,Esper Origins // Summon: Esper Maduin,2.0,NaN,[G],NaN,Sorcery // Enchantment Creature — Saga Elemental,NaN,NaN,NaN,"[Surveil, Transform, Flashback]",False,NaN,NaN,NaN,NaN,[G],False,legal,1.03,NaN,5434.0,False,True,False,True,False,True,False,3
